## Data Preprocessing

In [1]:
# import libraries
import pandas as pd
import numpy as np

In [2]:
# load dataset
df = pd.read_csv('fraud_detection_dataset.csv')
df = df.drop(columns=['timestamp', 'user_id'], errors='ignore')
print(df.head())

   amount         location device_type  is_fraud  age     income      debt  \
0  998.99        Grantfurt      Mobile         0   56   42524.98   8394.05   
1  241.39  Kimberlychester      Tablet         0   52   69884.04  28434.06   
2  836.42   Gutierrezville     Desktop         0   58  126953.62  39121.78   
3  612.74         Markside     Desktop         0   19  128650.70  39652.48   
4  135.55     Anthonyshire      Tablet         0   59  102020.39   7439.81   

   credit_score  
0           655  
1           395  
2           496  
3           612  
4           302  


In [3]:
print(f'Dataset shape: {df.shape}\n')
print(f'Missing Values:\n{df.isnull().sum()}\n')
print(f'Statistics:\n{df.describe(include="all")}\n')
print(f'Class counts:\n{df["is_fraud"].value_counts()}')

Dataset shape: (2000000, 8)

Missing Values:
amount          0
location        0
device_type     0
is_fraud        0
age             0
income          0
debt            0
credit_score    0
dtype: int64

Statistics:
              amount      location device_type   is_fraud           age  \
count   2.000000e+06       2000000     2000000  2000000.0  2.000000e+06   
unique           NaN        104592           3        NaN           NaN   
top              NaN  East Michael      Tablet        NaN           NaN   
freq             NaN          1713      668072        NaN           NaN   
mean    1.751680e+03           NaN         NaN        0.5  4.400430e+01   
std     1.504160e+03           NaN         NaN        0.5  1.529754e+01   
min     1.000000e+01           NaN         NaN        0.0  1.800000e+01   
25%     5.043400e+02           NaN         NaN        0.0  3.100000e+01   
50%     1.000005e+03           NaN         NaN        0.5  4.400000e+01   
75%     2.996710e+03           NaN 

In [4]:
# matrix of features (X) and independent variable (y)
X = df.drop(columns=['is_fraud'])
y = df['is_fraud']

In [5]:
# train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

In [6]:
# preprocessing pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

num_cols = list(X.select_dtypes(exclude='object').columns)
cat_cols = list(X.select_dtypes(include='object').columns)

num_pipe = Pipeline([
    ('scaler', StandardScaler())
])

cat_pipe = OneHotEncoder(handle_unknown='ignore')

ct = ColumnTransformer(transformers=[
    ('num_pipeline', num_pipe, num_cols),
    ('cat_pipeline', cat_pipe, cat_cols)
])

In [7]:
# XGBoost pipeline with GridSearchCV hyperparameter tuning
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from xgboost import XGBClassifier

xgboost_pipeline = Pipeline([
    ('preprocessor', ct),
    ('model', XGBClassifier(
        eval_metric='logloss',
        random_state=0,
        n_jobs=-1
    ))
])

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5],
    'model__learning_rate': [0.05, 0.1]
}

cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

grid_search = GridSearchCV(
    estimator=xgboost_pipeline,
    param_grid=param_grid,
    scoring='f1',
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f'Best cross-validation F1-score: {grid_search.best_score_:.4f}')
print(f'Best parameters: {grid_search.best_params_}')

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best cross-validation F1-score: 1.0000
Best parameters: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100}


In [8]:
# evaluate the tuned XGBoost model on the test set
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

best_model = grid_search.best_estimator_
test_predictions = best_model.predict(X_test)

accuracy = accuracy_score(y_test, test_predictions)
precision = precision_score(y_test, test_predictions, zero_division=0)
recall = recall_score(y_test, test_predictions, zero_division=0)
f1 = f1_score(y_test, test_predictions, zero_division=0)

print(f'Test Accuracy: {accuracy:.4f}')
print(f'Train Accuracy: {best_model.score(X_train, y_train):.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-score: {f1:.4f}')

Test Accuracy: 0.9999
Train Accuracy: 0.9999
Precision: 0.9999
Recall: 1.0000
F1-score: 0.9999


In [9]:
# save pipeline
import joblib

joblib.dump(best_model, '../backend/app/ml/fraud_model.pkl')

['../backend/app/ml/fraud_model.pkl']